# Chapter 8 Computational Lab
## Discrete Distributions and Count Models

This notebook accompanies Chapter 8 of *Probability Theory with Python and AI*.

The chapter applies the expectation theory of Chapter 7 to probability laws concentrated on finite or countable sets. It develops probability mass functions, moments and dispersion, then studies the main discrete models used throughout probability.

### Learning goals

By the end of the lab you should be able to:

1. distinguish a **countably valued random variable** from a variable with a **discrete law**;
2. construct and validate probability mass functions;
3. recover a cdf and expectations from a pmf;
4. compute raw, central and factorial moments;
5. use variance, coefficient of variation and dispersion index;
6. recognize discrete uniform and Bernoulli laws;
7. derive and compute binomial probabilities from independent Bernoulli trials;
8. use the geometric waiting-time convention and memoryless property;
9. work with the negative binomial convention used in this book;
10. analyze hypergeometric sampling without replacement;
11. use Poisson recursions, modes and equidispersion;
12. understand the Poisson limit of the binomial law;
13. use probability generating functions and their derivatives;
14. distinguish underdispersion, equidispersion and overdispersion;
15. select count models from mechanism and structure rather than moment matching alone;
16. audit AI-generated claims about discrete count models.

> **Parameterization rule.** For geometric and negative binomial laws, always state what is being counted. In this chapter, $\operatorname{Geom}(p)$ counts failures before the first success and $\operatorname{NB}(r,p)$ counts failures before the $r$-th success.


## 0. Setup

The notebook avoids probability-library pmf and cdf functions. The basic formulas are implemented directly so that every numerical result remains tied to the mathematics of the chapter.


In [ ]:
from fractions import Fraction
from math import comb, exp, factorial, lgamma, floor
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import HTML, Math, Markdown, clear_output, display

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


def fmt_fraction(x):
    x = Fraction(x)
    if x.denominator == 1:
        return str(x.numerator)
    return rf"\frac{{{x.numerator}}}{{{x.denominator}}}"


def discrete_uniform_pmf(k, m):
    if not isinstance(m, int) or isinstance(m, bool) or m < 1:
        raise ValueError("m must be a positive integer.")
    return 1 / m if isinstance(k, int) and 1 <= k <= m else 0.0


def bernoulli_pmf(k, p):
    if not 0 <= p <= 1:
        raise ValueError("p must lie in [0,1].")
    if k == 0:
        return 1 - p
    if k == 1:
        return p
    return 0.0


def binomial_pmf(k, n, p):
    if not isinstance(n, int) or isinstance(n, bool) or n < 1:
        raise ValueError("n must be a positive integer.")
    if not 0 <= p <= 1:
        raise ValueError("p must lie in [0,1].")
    if not isinstance(k, int) or k < 0 or k > n:
        return 0.0
    return comb(n, k) * p**k * (1-p)**(n-k)


def geometric_pmf(k, p):
    if not 0 < p < 1:
        raise ValueError("p must lie in (0,1).")
    if not isinstance(k, int) or k < 0:
        return 0.0
    return p * (1-p)**k


def negative_binomial_pmf(k, r, p):
    if not isinstance(r, int) or isinstance(r, bool) or r < 1:
        raise ValueError("r must be a positive integer.")
    if not 0 < p < 1:
        raise ValueError("p must lie in (0,1).")
    if not isinstance(k, int) or k < 0:
        return 0.0
    return comb(k+r-1, k) * p**r * (1-p)**k


def hypergeometric_pmf(k, N, K, n):
    if (
        not all(isinstance(v, int) and not isinstance(v, bool) for v in (N, K, n))
        or N < 1
        or K < 0
        or n < 0
        or K > N
        or n > N
    ):
        raise ValueError("Require N>=1 and 0<=K,n<=N, all integers.")

    lower = max(0, n-(N-K))
    upper = min(n, K)

    if not isinstance(k, int) or k < lower or k > upper:
        return 0.0

    return comb(K, k) * comb(N-K, n-k) / comb(N, n)


def poisson_pmf(k, lam):
    if isinstance(lam, bool) or lam < 0:
        raise ValueError("lambda must be a non-negative real number.")
    if not isinstance(k, int) or k < 0:
        return 0.0
    if lam == 0:
        return 1.0 if k == 0 else 0.0
    return exp(-lam) * lam**k / factorial(k)


def moments(values, probs):
    x = np.asarray(values, dtype=float)
    p = np.asarray(probs, dtype=float)
    total = np.sum(p)
    mean = np.sum(x*p)
    variance = np.sum((x-mean)**2 * p)
    return total, mean, variance


def pgf_from_table(values, probs, s):
    x = np.asarray(values, dtype=int)
    p = np.asarray(probs, dtype=float)
    return np.sum(p * (s**x))


def cdf_from_table(values, probs):
    order = np.argsort(values)
    x = np.asarray(values)[order]
    p = np.asarray(probs, dtype=float)[order]
    return x, np.cumsum(p)


def show_result(title, *latex_lines, note=None):
    display(HTML(
        f"<div style='border-left:5px solid;padding:8px 12px;margin:8px 0'>"
        f"<b>{title}</b></div>"
    ))
    for line in latex_lines:
        display(Math(line))
    if note:
        display(Markdown(note))


display(HTML(
    "<div style='padding:10px;border:1px solid'>"
    "<b>Setup complete.</b> Discrete-distribution tools are ready."
    "</div>"
))


## 1. Countably valued variables and discrete laws

A random variable $X$ is **countably valued** if its pointwise range $X(\Omega)$ is finite or countably infinite.

It has a **discrete law** if there exists a finite or countable set $D\subseteq\mathbb R$ such that

$$
P(X\in D)=1.
$$

The distinction matters:

- countable-valuedness is a pointwise property of the mapping;
- discreteness is a property of the probability law and is unchanged by almost-sure modification.

Every countably valued random variable has a discrete law. Conversely, every variable with a discrete law has an almost-surely equal countably valued version.


### The same measurable map under two probability laws

Let

$$
\Omega=[0,1],
\qquad
X(\omega)=\omega.
$$

The map is not countably valued because its range is $[0,1]$.

Under the point-mass probability $P=\delta_0$,

$$
P(X=0)=1,
$$

so its law is discrete.

Under uniform probability on $[0,1]$, every countable set has probability zero, so the law is not discrete.


In [ ]:
grid = np.linspace(0, 1, 500)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(grid, grid)
ax.set_xlabel("omega")
ax.set_ylabel("X(omega)")
ax.set_title("The same map X(omega)=omega can have different laws under different P")
plt.show()

display(Markdown(
    "The graph of the map does not change when the probability measure changes; "
    "the induced law does."
))


### Pmf support versus topological support

If

$$
P(X=1/n)=2^{-n},
\qquad
n=1,2,\ldots,
$$

then the pmf support is

$$
\mathcal S_X=\{1,1/2,1/3,\ldots\}.
$$

Its topological support also contains the limit point $0$.

Thus the set of positive-mass points need not be closed.


In [ ]:
support_N = widgets.IntSlider(value=12, min=3, max=30, description="N")
support_output = widgets.Output()


def update_support(*_):
    with support_output:
        clear_output(wait=True)

        N = support_N.value
        ns = np.arange(1, N+1)
        xs = 1/ns
        ps = 2.0**(-ns)

        fig, ax = plt.subplots(figsize=(8, 3.2))
        ax.scatter(xs, ps)
        ax.axvline(0, linestyle="--")
        ax.set_xlabel("support point 1/n")
        ax.set_ylabel("mass 2^{-n}")
        ax.set_title("Positive-mass points accumulate at 0")
        plt.show()


support_N.observe(update_support, names="value")
display(widgets.VBox([support_N, support_output]))
update_support()


## 2. Probability mass functions

For a discrete law, the probability mass function is

$$
p_X(x)=P(X=x).
$$

A non-negative function $p$ on a finite or countable set $D$ defines a probability law exactly when

$$
\sum_{x\in D}p(x)=1.
$$

Then, for every Borel set $A$,

$$
P(X\in A)
=
\sum_{x\in A\cap D}p(x).
$$


### A geometric sequence of proposed masses

On

$$
D=\{0,1,2,\ldots\},
$$

let

$$
p(k)=2^{-(k+1)}.
$$

Since

$$
\sum_{k=0}^{\infty}2^{-(k+1)}=1,
$$

this is a valid pmf.

The probability of an even value is

$$
P(X\text{ even})
=
\sum_{j=0}^{\infty}2^{-(2j+1)}
=
\frac23.
$$


In [ ]:
partial_N = widgets.IntSlider(value=12, min=1, max=40, description="N")
partial_output = widgets.Output()


def update_geometric_mass(*_):
    with partial_output:
        clear_output(wait=True)

        N = partial_N.value
        partial = sum(2**(-(k+1)) for k in range(N+1))
        even_partial = sum(2**(-(2*j+1)) for j in range((N//2)+1))

        display(Math(
            r"\sum_{k=0}^{" + str(N) + r"}2^{-(k+1)}="
            + f"{partial:.8f}"
        ))
        display(Math(
            r"\text{partial even-mass sum}="
            + f"{even_partial:.8f}"
        ))
        display(Math(r"P(X\text{ even})=\frac23"))


partial_N.observe(update_geometric_mass, names="value")
display(widgets.VBox([partial_N, partial_output]))
update_geometric_mass()


## 3. Cdf and expectation from a pmf

For a discrete law,

$$
\boxed{
F_X(t)
=
\sum_{\substack{x\in\mathcal S_X\\x\le t}}
p_X(x).
}
$$

The jump recovers the mass:

$$
\boxed{
p_X(x)
=
F_X(x)-F_X(x-).
}
$$

For Borel $g$,

$$
\boxed{
\mathbb E[g(X)]
=
\sum_{x\in\mathcal S_X}
g(x)p_X(x),
}
$$

whenever the expectation is defined.


### Finite count example

Suppose

$$
\begin{array}{c|ccccc}
n&0&1&2&3&4\\
\hline
p_N(n)&0.40&0.30&0.18&0.09&0.03.
\end{array}
$$

Then

$$
F_N(2)=0.88,
$$

$$
P(N>2)=0.12,
$$

and

$$
\mathbb E[N]=1.05.
$$


In [ ]:
finite_values = np.array([0,1,2,3,4])
finite_probs = np.array([0.40,0.30,0.18,0.09,0.03])

x, F = cdf_from_table(finite_values, finite_probs)
total, mean, variance = moments(finite_values, finite_probs)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.step(x, F, where="post")
ax.scatter(x, F)
ax.set_ylim(0, 1.03)
ax.set_xlabel("n")
ax.set_ylabel("F_N(n)")
ax.set_title("CDF from a finite pmf")
plt.show()

display(Math(r"F_N(2)=0.88"))
display(Math(r"P(N>2)=0.12"))
display(Math(r"\mathbb E[N]=" + f"{mean:.2f}"))


## 4. Moments and measures of dispersion

For integer $r\ge1$:

$$
\mu_r'
=
\mathbb E[X^r]
$$

is the $r$-th raw moment, and

$$
\mu_r
=
\mathbb E[(X-\mu)^r],
\qquad
\mu=\mathbb E[X],
$$

is the $r$-th central moment when defined.

Variance is

$$
\operatorname{Var}(X)
=
\mathbb E[(X-\mathbb E[X])^2].
$$

The computational identity is

$$
\boxed{
\operatorname{Var}(X)
=
\mathbb E[X^2]
-
(\mathbb E[X])^2.
}
$$


### Affine transformations

For real $a,b$,

$$
\boxed{
\operatorname{Var}(aX+b)
=
a^2\operatorname{Var}(X).
}
$$

A shift changes location but not variance. Multiplying by $a$ multiplies standard deviation by $|a|$ and variance by $a^2$.


In [ ]:
affine_a = widgets.FloatSlider(value=3, min=-5, max=5, step=0.5, description="a")
affine_b = widgets.FloatSlider(value=-4, min=-10, max=10, step=1, description="b")
affine_output = widgets.Output()


def update_affine(*_):
    with affine_output:
        clear_output(wait=True)

        a = affine_a.value
        b = affine_b.value
        p = 0.5

        var_X = p*(1-p)
        var_Y = (a**2)*var_X

        display(Math(r"\operatorname{Var}(X)=\frac14"))
        display(Math(
            r"\operatorname{Var}(aX+b)=a^2\operatorname{Var}(X)="
            + f"{var_Y:.6f}"
        ))


for control in (affine_a, affine_b):
    control.observe(update_affine, names="value")

display(widgets.VBox([
    widgets.HBox([affine_a, affine_b]),
    affine_output,
]))
update_affine()


### Factorial moments

For non-negative integer-valued $X$, define the falling factorial

$$
(X)_r
=
X(X-1)\cdots(X-r+1),
$$

with $(X)_0=1$.

The first two factorial moments satisfy

$$
\mathbb E[(X)_1]=\mathbb E[X],
$$

$$
\mathbb E[(X)_2]
=
\mathbb E[X^2]-\mathbb E[X].
$$

Therefore

$$
\boxed{
\operatorname{Var}(X)
=
\mathbb E[(X)_2]
+
\mathbb E[X]
-
(\mathbb E[X])^2.
}
$$


In [ ]:
factorial_values = np.array([0,1,2])
factorial_probs = np.array([0.5,0.3,0.2])

mean = np.sum(factorial_values*factorial_probs)
second_fact = np.sum(
    factorial_values*(factorial_values-1)*factorial_probs
)
variance = second_fact + mean - mean**2

display(Math(r"\mathbb E[N]=" + f"{mean:.2f}"))
display(Math(r"\mathbb E[(N)_2]=" + f"{second_fact:.2f}"))
display(Math(r"\operatorname{Var}(N)=" + f"{variance:.2f}"))


### Dispersion index

For a non-negative integer-valued count $N$ with positive finite mean,

$$
\boxed{
D_N
=
\frac{\operatorname{Var}(N)}{\mathbb E[N]}.
}
$$

The count is:

- **underdispersed** if $D_N<1$;
- **equidispersed** if $D_N=1$;
- **overdispersed** if $D_N>1$.

This is a diagnostic, not a complete model-selection rule.


In [ ]:
disp_mean = widgets.FloatSlider(value=8, min=0.1, max=20, step=0.1, description="mean")
disp_var = widgets.FloatSlider(value=16, min=0, max=40, step=0.1, description="variance")
disp_output = widgets.Output()


def update_dispersion(*_):
    with disp_output:
        clear_output(wait=True)

        D = disp_var.value / disp_mean.value

        if D < 1 - 1e-9:
            label = "underdispersed"
        elif D > 1 + 1e-9:
            label = "overdispersed"
        else:
            label = "equidispersed"

        display(Math(r"D_N=" + f"{D:.4f}"))
        display(Markdown(f"**Diagnostic:** {label}"))


for control in (disp_mean, disp_var):
    control.observe(update_dispersion, names="value")

display(widgets.VBox([
    widgets.HBox([disp_mean, disp_var]),
    disp_output,
]))
update_dispersion()


## 5. Discrete uniform distribution

For integer $m\ge1$,

$$
X\sim\operatorname{DU}\{1,\ldots,m\}
$$

means

$$
P(X=k)=\frac1m,
\qquad
k=1,\ldots,m.
$$

Its moments are

$$
\boxed{
\mathbb E[X]=\frac{m+1}{2},
\qquad
\operatorname{Var}(X)=\frac{m^2-1}{12}.
}
$$


In [ ]:
du_m = widgets.IntSlider(value=5, min=1, max=20, description="m")
du_output = widgets.Output()


def update_du(*_):
    with du_output:
        clear_output(wait=True)

        m = du_m.value
        ks = np.arange(1, m+1)
        probs = np.full(m, 1/m)

        total, mean, variance = moments(ks, probs)

        display(Math(r"\mathbb E[X]=" + f"{mean:.6f}"))
        display(Math(r"\operatorname{Var}(X)=" + f"{variance:.6f}"))

        fig, ax = plt.subplots(figsize=(8, 3.0))
        ax.bar(ks, probs)
        ax.set_xlabel("k")
        ax.set_ylabel("P(X=k)")
        ax.set_title("Discrete uniform pmf")
        plt.show()


du_m.observe(update_du, names="value")
display(widgets.VBox([du_m, du_output]))
update_du()


## 6. Bernoulli distribution

For $0\le p\le1$,

$$
X\sim\operatorname{Bernoulli}(p)
$$

means

$$
P(X=1)=p,
\qquad
P(X=0)=1-p.
$$

For an event $A$,

$$
\mathbf 1_A
\sim
\operatorname{Bernoulli}(P(A)).
$$

The moments are

$$
\boxed{
\mathbb E[X]=p,
\qquad
\operatorname{Var}(X)=p(1-p).
}
$$


In [ ]:
bern_p = widgets.FloatSlider(value=0.35, min=0, max=1, step=0.01, description="p")
bern_output = widgets.Output()


def update_bernoulli(*_):
    with bern_output:
        clear_output(wait=True)

        p = bern_p.value

        display(Math(r"\mathbb E[X]=" + f"{p:.4f}"))
        display(Math(
            r"\operatorname{Var}(X)=" + f"{p*(1-p):.4f}"
        ))

        fig, ax = plt.subplots(figsize=(6, 3))
        ax.bar([0,1], [1-p,p])
        ax.set_xticks([0,1])
        ax.set_ylabel("mass")
        ax.set_title("Bernoulli pmf")
        plt.show()


bern_p.observe(update_bernoulli, names="value")
display(widgets.VBox([bern_p, bern_output]))
update_bernoulli()


## 7. Binomial distribution

The binomial model counts successes in a fixed number of homogeneous mutually independent Bernoulli trials.

If

$$
X=\sum_{i=1}^{n}\mathbf 1_{A_i},
$$

where the events $A_i$ are mutually independent and

$$
P(A_i)=p,
$$

then

$$
\boxed{
X\sim\operatorname{Bin}(n,p).
}
$$

Its pmf is

$$
\boxed{
P(X=k)
=
\binom nkp^k(1-p)^{n-k},
\qquad
k=0,\ldots,n.
}
$$


### Why mutual independence matters

For a specified pattern with success indices $J$ and failure indices $L$,

$$
P\left(
\bigcap_{j\in J}A_j
\cap
\bigcap_{\ell\in L}A_\ell^c
\right)
=
\prod_{j\in J}P(A_j)
\prod_{\ell\in L}(1-P(A_\ell)).
$$

Pairwise independence alone does not justify an arbitrary multi-event pattern product.


In [ ]:
pattern_probability = 0.2 * (1-0.3) * 0.4
display(Math(
    r"P(A_1\cap A_2^c\cap A_3)=" + f"{pattern_probability:.3f}"
))


### Binomial moments

If

$$
X\sim\operatorname{Bin}(n,p),
$$

then

$$
\mathbb E[X]=np,
$$

$$
\mathbb E[(X)_2]=n(n-1)p^2,
$$

and

$$
\boxed{
\operatorname{Var}(X)=np(1-p).
}
$$

For $0<p<1$,

$$
D_X=1-p<1,
$$

so the non-degenerate binomial law is underdispersed.


In [ ]:
bin_n = widgets.IntSlider(value=20, min=1, max=100, description="n")
bin_p = widgets.FloatSlider(value=0.10, min=0, max=1, step=0.01, description="p")
bin_output = widgets.Output()


def update_binomial(*_):
    with bin_output:
        clear_output(wait=True)

        n = bin_n.value
        p = bin_p.value
        ks = np.arange(n+1)
        probs = np.array([binomial_pmf(int(k), n, p) for k in ks])

        total, mean, variance = moments(ks, probs)

        display(Math(r"\sum_kp(k)=" + f"{total:.12f}"))
        display(Math(r"\mathbb E[X]=" + f"{mean:.6f}"))
        display(Math(r"\operatorname{Var}(X)=" + f"{variance:.6f}"))

        fig, ax = plt.subplots(figsize=(8, 3.5))
        ax.stem(ks, probs)
        ax.set_xlabel("k")
        ax.set_ylabel("P(X=k)")
        ax.set_title("Binomial pmf")
        plt.show()


for control in (bin_n, bin_p):
    control.observe(update_binomial, names="value")

display(widgets.VBox([
    widgets.HBox([bin_n, bin_p]),
    bin_output,
]))
update_binomial()


## 8. Historical problem: Newton--Pepys dice

In 1693 Samuel Pepys asked Isaac Newton to compare:

1. at least one six among $6$ fair dice;
2. at least two sixes among $12$ fair dice;
3. at least three sixes among $18$ fair dice.

Let

$$
X_r\sim\operatorname{Bin}(6r,1/6).
$$

The probabilities are approximately

$$
P(X_1\ge1)\approx0.665102,
$$

$$
P(X_2\ge2)\approx0.618667,
$$

$$
P(X_3\ge3)\approx0.597346.
$$

Thus the first choice is best. Scaling both the number of dice and the target count does not preserve the tail probability.


In [ ]:
pepys = []

for r in (1,2,3):
    n = 6*r
    p = 1/6
    prob = sum(
        binomial_pmf(k, n, p)
        for k in range(r, n+1)
    )
    pepys.append(prob)

for r, prob in enumerate(pepys, 1):
    display(Math(
        r"P(X_" + str(r) + r"\ge" + str(r) + r")="
        + f"{prob:.6f}"
    ))

display(Markdown(
    f"Ordering verified: **{pepys[0] > pepys[1] > pepys[2]}**"
))


## 9. Geometric distribution

The chapter uses the convention:

$$
X
=
\text{number of failures before the first success}.
$$

For $0<p<1$ and $q=1-p$,

$$
\boxed{
P(X=k)=pq^k,
\qquad
k=0,1,2,\ldots.
}
$$

The total number of trials is

$$
T=X+1.
$$


### Tail and moments

For $m\ge0$,

$$
P(X\ge m)=q^m.
$$

Also,

$$
\boxed{
\mathbb E[X]=\frac{q}{p},
\qquad
\operatorname{Var}(X)=\frac{q}{p^2}.
}
$$

Consequently,

$$
\mathbb E[T]=\frac1p.
$$


In [ ]:
geom_p = widgets.FloatSlider(value=0.25, min=0.01, max=0.99, step=0.01, description="p")
geom_m = widgets.IntSlider(value=5, min=0, max=20, description="m")
geom_output = widgets.Output()


def update_geometric(*_):
    with geom_output:
        clear_output(wait=True)

        p = geom_p.value
        q = 1-p
        m = geom_m.value

        display(Math(
            r"P(X\ge m)=" + f"{q**m:.6f}"
        ))
        display(Math(
            r"\mathbb E[X]=" + f"{q/p:.6f}"
        ))
        display(Math(
            r"\operatorname{Var}(X)=" + f"{q/(p**2):.6f}"
        ))
        display(Math(
            r"\mathbb E[T]=" + f"{1/p:.6f}"
        ))


for control in (geom_p, geom_m):
    control.observe(update_geometric, names="value")

display(widgets.VBox([
    widgets.HBox([geom_p, geom_m]),
    geom_output,
]))
update_geometric()


### Memoryless property

For all integers $m,n\ge0$,

$$
\boxed{
P(X\ge m+n\mid X\ge m)
=
P(X\ge n).
}
$$

Among non-negative integer-valued waiting times, this property characterizes the geometric law, apart from the degenerate endpoint.


In [ ]:
mem_p = widgets.FloatSlider(value=0.20, min=0.05, max=0.95, step=0.05, description="p")
mem_m = widgets.IntSlider(value=5, min=0, max=20, description="m")
mem_n = widgets.IntSlider(value=3, min=0, max=20, description="n")
mem_output = widgets.Output()


def update_memoryless(*_):
    with mem_output:
        clear_output(wait=True)

        p = mem_p.value
        q = 1-p
        m = mem_m.value
        n = mem_n.value

        conditional = q**(m+n) / q**m
        direct = q**n

        display(Math(
            r"P(X\ge m+n\mid X\ge m)=" + f"{conditional:.6f}"
        ))
        display(Math(
            r"P(X\ge n)=" + f"{direct:.6f}"
        ))


for control in (mem_p, mem_m, mem_n):
    control.observe(update_memoryless, names="value")

display(widgets.VBox([
    widgets.HBox([mem_p, mem_m, mem_n]),
    mem_output,
]))
update_memoryless()


## 10. Negative binomial distribution

For integer $r\ge1$ and $0<p<1$,

$$
X\sim\operatorname{NB}(r,p)
$$

means that $X$ counts failures before the $r$-th success, with pmf

$$
\boxed{
P(X=k)
=
\binom{k+r-1}{k}
p^r(1-p)^k,
\qquad
k=0,1,2,\ldots.
}
$$

The parameterization must always be stated explicitly because other conventions are common.


### Negative binomial series

For integer $r\ge1$ and $|z|<1$,

$$
\boxed{
\sum_{k=0}^{\infty}
\binom{k+r-1}{k}z^k
=
(1-z)^{-r}.
}
$$

Setting $z=1-p$ proves normalization of the negative binomial pmf.


In [ ]:
nb_r = widgets.IntSlider(value=4, min=1, max=12, description="r")
nb_p = widgets.FloatSlider(value=0.50, min=0.05, max=0.95, step=0.05, description="p")
nb_output = widgets.Output()


def update_nb(*_):
    with nb_output:
        clear_output(wait=True)

        r = nb_r.value
        p = nb_p.value
        q = 1-p

        mean = r*q/p
        variance = r*q/(p**2)
        D = variance/mean if mean > 0 else float("nan")

        display(Math(r"\mathbb E[X]=" + f"{mean:.6f}"))
        display(Math(r"\operatorname{Var}(X)=" + f"{variance:.6f}"))
        display(Math(r"D_X=" + f"{D:.6f}"))

        ks = np.arange(0, 30)
        probs = np.array([
            negative_binomial_pmf(int(k), r, p)
            for k in ks
        ])

        fig, ax = plt.subplots(figsize=(8, 3.5))
        ax.stem(ks, probs)
        ax.set_xlabel("failures k")
        ax.set_ylabel("mass")
        ax.set_title("Negative binomial pmf")
        plt.show()


for control in (nb_r, nb_p):
    control.observe(update_nb, names="value")

display(widgets.VBox([
    widgets.HBox([nb_r, nb_p]),
    nb_output,
]))
update_nb()


### Geometric as a special case

Under the conventions of this chapter,

$$
\boxed{
\operatorname{NB}(1,p)
=
\operatorname{Geom}(p).
}
$$


In [ ]:
p = 0.2
checks = [
    abs(negative_binomial_pmf(k, 1, p) - geometric_pmf(k, p))
    for k in range(10)
]

display(Markdown(
    f"First ten masses agree numerically: **{max(checks) < 1e-14}**"
))


### Moment calibration

For a target mean $m>0$ and variance $v>m$, moment matching gives

$$
\boxed{
p=\frac{m}{v},
\qquad
r=\frac{m^2}{v-m}.
}
$$

Under the waiting-time definition used here, $r$ must be a positive integer.


In [ ]:
cal_m = widgets.FloatSlider(value=8, min=0.5, max=20, step=0.5, description="mean")
cal_v = widgets.FloatSlider(value=16, min=0.5, max=40, step=0.5, description="variance")
cal_output = widgets.Output()


def update_nb_calibration(*_):
    with cal_output:
        clear_output(wait=True)

        m = cal_m.value
        v = cal_v.value

        if v <= m:
            display(Markdown("**Negative binomial moment matching requires variance > mean.**"))
            return

        p = m/v
        r = m*m/(v-m)

        display(Math(r"p=" + f"{p:.6f}"))
        display(Math(r"r=" + f"{r:.6f}"))
        display(Markdown(
            f"Integer-r waiting-time parameterization compatible: "
            f"**{abs(r-round(r)) < 1e-10}**"
        ))


for control in (cal_m, cal_v):
    control.observe(update_nb_calibration, names="value")

display(widgets.VBox([
    widgets.HBox([cal_m, cal_v]),
    cal_output,
]))
update_nb_calibration()


## 11. Hypergeometric distribution

A population of size $N$ contains $K$ specified-type objects. A sample of $n$ distinct objects is selected uniformly **without replacement**.

If $X$ counts the specified-type objects in the sample, then

$$
X\sim\operatorname{Hyp}(N,K,n)
$$

with

$$
\boxed{
P(X=k)
=
\frac{
\binom Kk
\binom{N-K}{n-k}
}{
\binom Nn
}.
}
$$

The support satisfies

$$
\max\{0,n-(N-K)\}
\le k\le
\min\{n,K\}.
$$


### Hypergeometric moments

For $N\ge2$,

$$
\boxed{
\mathbb E[X]
=
n\frac KN,
}
$$

and

$$
\boxed{
\operatorname{Var}(X)
=
n\frac KN
\left(1-\frac KN\right)
\frac{N-n}{N-1}.
}
$$

The factor

$$
\frac{N-n}{N-1}
$$

is the finite population correction.


In [ ]:
hyp_N = widgets.IntSlider(value=100, min=2, max=300, description="N")
hyp_K = widgets.IntSlider(value=20, min=0, max=100, description="K")
hyp_n = widgets.IntSlider(value=10, min=0, max=100, description="n")
hyp_output = widgets.Output()


def update_hypergeometric(*_):
    with hyp_output:
        clear_output(wait=True)

        N = hyp_N.value
        K = hyp_K.value
        n = hyp_n.value

        if K > N or n > N:
            display(Markdown("**Require K<=N and n<=N.**"))
            return

        lower = max(0, n-(N-K))
        upper = min(n, K)
        ks = np.arange(lower, upper+1)

        probs = np.array([
            hypergeometric_pmf(int(k), N, K, n)
            for k in ks
        ])

        total, mean, variance = moments(ks, probs)

        theory_mean = n*K/N
        theory_var = (
            0.0 if N == 1
            else n*(K/N)*(1-K/N)*(N-n)/(N-1)
        )

        display(Math(r"\sum_kp(k)=" + f"{total:.12f}"))
        display(Math(r"\mathbb E[X]=" + f"{mean:.6f}"))
        display(Math(r"\operatorname{Var}(X)=" + f"{variance:.6f}"))
        display(Math(r"\text{theoretical variance}=" + f"{theory_var:.6f}"))

        fig, ax = plt.subplots(figsize=(8, 3.5))
        ax.stem(ks, probs)
        ax.set_xlabel("k")
        ax.set_ylabel("mass")
        ax.set_title("Hypergeometric pmf")
        plt.show()


for control in (hyp_N, hyp_K, hyp_n):
    control.observe(update_hypergeometric, names="value")

display(widgets.VBox([
    widgets.HBox([hyp_N, hyp_K, hyp_n]),
    hyp_output,
]))
update_hypergeometric()


### Without replacement versus with replacement

A binomial model with success probability $K/N$ would have variance

$$
n\frac KN\left(1-\frac KN\right).
$$

Hypergeometric sampling multiplies this by

$$
\frac{N-n}{N-1}\le1.
$$

Sampling without replacement therefore reduces variability.


## 12. Poisson distribution

For $\lambda=0$, the Poisson law is the point mass at zero.

For $\lambda>0$,

$$
\boxed{
P(X=k)
=
e^{-\lambda}\frac{\lambda^k}{k!},
\qquad
k=0,1,2,\ldots.
}
$$

Its mean and variance are

$$
\boxed{
\mathbb E[X]
=
\operatorname{Var}(X)
=
\lambda.
}
$$

Thus the positive-mean Poisson law is equidispersed.


### Consecutive-mass recursion

For $\lambda>0$,

$$
p_X(0)=e^{-\lambda},
$$

and

$$
\boxed{
p_X(k+1)
=
\frac{\lambda}{k+1}p_X(k).
}
$$

For large $\lambda$, starting from $e^{-\lambda}$ can underflow numerically. A stable implementation should evaluate a mass near the mode using logarithms and recurse in both directions.


In [ ]:
pois_lam = widgets.FloatSlider(value=4.0, min=0.0, max=30.0, step=0.5, description="lambda")
pois_output = widgets.Output()


def update_poisson(*_):
    with pois_output:
        clear_output(wait=True)

        lam = pois_lam.value
        max_k = max(12, int(lam + 6*math.sqrt(max(lam, 1))))
        ks = np.arange(0, max_k+1)
        probs = np.array([
            poisson_pmf(int(k), lam)
            for k in ks
        ])

        total, mean, variance = moments(ks, probs)

        display(Math(r"\text{truncated total mass}=" + f"{total:.12f}"))
        display(Math(r"\text{truncated mean}=" + f"{mean:.6f}"))
        display(Math(r"\text{truncated variance}=" + f"{variance:.6f}"))
        display(Math(r"\text{theory: mean=variance}=" + f"{lam:.6f}"))

        fig, ax = plt.subplots(figsize=(8, 3.5))
        ax.stem(ks, probs)
        ax.set_xlabel("k")
        ax.set_ylabel("mass")
        ax.set_title("Poisson pmf")
        plt.show()


pois_lam.observe(update_poisson, names="value")
display(widgets.VBox([pois_lam, pois_output]))
update_poisson()


### Poisson mode

If $\lambda>0$ is not an integer, the unique mode is

$$
\lfloor\lambda\rfloor.
$$

If $\lambda$ is a positive integer, there are two modes:

$$
\lambda-1
\qquad\text{and}\qquad
\lambda.
$$

This follows directly from the adjacent-mass ratio.


In [ ]:
mode_lam = widgets.FloatSlider(value=4.0, min=0.1, max=12, step=0.1, description="lambda")
mode_output = widgets.Output()


def update_mode(*_):
    with mode_output:
        clear_output(wait=True)

        lam = mode_lam.value

        if abs(lam-round(lam)) < 1e-10:
            n = int(round(lam))
            modes = [n-1, n]
        else:
            modes = [math.floor(lam)]

        display(Markdown(f"**Mode(s): {modes}**"))


mode_lam.observe(update_mode, names="value")
display(widgets.VBox([mode_lam, mode_output]))
update_mode()


## 13. Poisson limit of the binomial distribution

Fix $\lambda>0$ and let

$$
X_n
\sim
\operatorname{Bin}\left(n,\frac{\lambda}{n}\right).
$$

For every fixed integer $k\ge0$,

$$
\boxed{
P(X_n=k)
\longrightarrow
e^{-\lambda}\frac{\lambda^k}{k!}.
}
$$

Thus the binomial law converges to $\operatorname{Poisson}(\lambda)$ in the rare-event regime

$$
n\to\infty,
\qquad
p_n=\frac{\lambda}{n}\to0,
\qquad
np_n=\lambda.
$$


### What the theorem does not say

For finite $n$,

$$
\operatorname{Bin}(n,p)
$$

and

$$
\operatorname{Poisson}(np)
$$

are generally **not identical**.

Matching the mean does not remove the finite-$n$ differences in the pmf or tails.


In [ ]:
limit_lam = widgets.FloatSlider(value=2.0, min=0.5, max=8.0, step=0.5, description="lambda")
limit_n = widgets.IntSlider(value=20, min=5, max=500, step=5, description="n")
limit_output = widgets.Output()


def update_poisson_limit(*_):
    with limit_output:
        clear_output(wait=True)

        lam = limit_lam.value
        n = limit_n.value

        if lam/n > 1:
            display(Markdown("**Need lambda/n <= 1. Increase n or decrease lambda.**"))
            return

        p = lam/n
        ks = np.arange(0, min(n, 15)+1)
        b = np.array([binomial_pmf(int(k), n, p) for k in ks])
        z = np.array([poisson_pmf(int(k), lam) for k in ks])

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(ks, b, marker="o", label="Binomial")
        ax.plot(ks, z, marker="s", linestyle="--", label="Poisson")
        ax.set_xlabel("k")
        ax.set_ylabel("mass")
        ax.set_title("Binomial rare-event law versus Poisson limit")
        ax.legend()
        plt.show()

        display(Math(
            r"|P(X_n=0)-P(Z=0)|="
            + f"{abs(b[0]-z[0]):.8f}"
        ))


for control in (limit_lam, limit_n):
    control.observe(update_poisson_limit, names="value")

display(widgets.VBox([
    widgets.HBox([limit_lam, limit_n]),
    limit_output,
]))
update_poisson_limit()


### Book comparison: $\operatorname{Bin}(20,0.1)$ versus $\operatorname{Poisson}(2)$

Both distributions have mean $2$, but their masses differ.

The approximation is useful, not exact.


In [ ]:
ks = np.arange(0, 11)
b = np.array([binomial_pmf(int(k), 20, 0.1) for k in ks])
z = np.array([poisson_pmf(int(k), 2.0) for k in ks])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ks, b, marker="o", label="Bin(20,0.1)")
ax.plot(ks, z, marker="s", linestyle="--", label="Poisson(2)")
ax.set_xlabel("count k")
ax.set_ylabel("probability mass")
ax.set_xticks(ks)
ax.legend()
ax.set_title("Finite binomial law and its Poisson approximation")
plt.show()

display(Math(r"P_{\mathrm{Bin}}(0)=" + f"{b[0]:.6f}"))
display(Math(r"P_{\mathrm{Pois}}(0)=" + f"{z[0]:.6f}"))


## 14. Probability generating functions

For a non-negative integer-valued random variable,

$$
\boxed{
G_X(s)
=
\mathbb E[s^X]
=
\sum_{k=0}^{\infty}p_X(k)s^k,
\qquad
|s|\le1.
}
$$

The pgf packages the entire law into one analytic object.


### Recovering probabilities from derivatives

For every integer $k\ge0$,

$$
\boxed{
p_X(k)
=
\frac{G_X^{(k)}(0)}{k!},
}
$$

where $G_X^{(k)}$ denotes the $k$-th derivative of $G_X$.

Therefore the pgf uniquely determines the law of a non-negative integer-valued random variable.


In [ ]:
# G(s)=(1+s)^3/8 is Bin(3,1/2)
pgf_coeffs = [
    comb(3,k) / 8
    for k in range(4)
]

display(Markdown(
    "Coefficients of $G(s)=(1+s)^3/8$: "
    f"**{pgf_coeffs}**"
))
display(Markdown(
    "These are exactly the pmf values of $\\operatorname{Bin}(3,1/2)$."
))


### Factorial moments from pgf derivatives

For integer $r\ge1$,

$$
G_X^{(r)}(s)
=
\sum_{k=r}^{\infty}
(k)_r p_X(k)s^{k-r}.
$$

Hence, using the left limit at $1$,

$$
\boxed{
G_X^{(r)}(1-)
=
\mathbb E[(X)_r],
}
$$

possibly with value $+\infty$.

In particular,

$$
\mathbb E[X]=G_X'(1-),
$$

and, when the second moment is finite,

$$
\boxed{
\operatorname{Var}(X)
=
G_X''(1-)
+
G_X'(1-)
-
\bigl(G_X'(1-)\bigr)^2.
}
$$


In [ ]:
# Binomial(3, 1/2)
n = 3
p = 0.5

G1 = n*p
G2 = n*(n-1)*(p**2)
variance = G2 + G1 - G1**2

display(Math(r"G_X'(1)=\frac32"))
display(Math(r"G_X''(1)=\frac32"))
display(Math(r"\operatorname{Var}(X)=" + f"{variance:.6f}"))


### Pgf formulas for principal laws

For the conventions of this chapter:

$$
G_{\mathrm{Bern}}(s)
=
1-p+ps,
$$

$$
G_{\mathrm{Bin}}(s)
=
(1-p+ps)^n,
$$

$$
G_{\mathrm{Geom}}(s)
=
\frac{p}{1-(1-p)s},
$$

$$
G_{\mathrm{NB}}(s)
=
\left(
\frac{p}{1-(1-p)s}
\right)^r,
$$

and

$$
G_{\mathrm{Pois}}(s)
=
e^{\lambda(s-1)}.
$$


In [ ]:
pgf_s = widgets.FloatSlider(value=0.7, min=0.0, max=1.0, step=0.01, description="s")
pgf_output = widgets.Output()


def update_pgf(*_):
    with pgf_output:
        clear_output(wait=True)

        s = pgf_s.value

        bern = 1-0.3+0.3*s
        binom = (1-0.2+0.2*s)**5
        geom = 0.4/(1-0.6*s)
        nb = (0.5/(1-0.5*s))**4
        pois = math.exp(3*(s-1))

        display(Math(r"G_{\mathrm{Bern}}(s)=" + f"{bern:.6f}"))
        display(Math(r"G_{\mathrm{Bin}}(s)=" + f"{binom:.6f}"))
        display(Math(r"G_{\mathrm{Geom}}(s)=" + f"{geom:.6f}"))
        display(Math(r"G_{\mathrm{NB}}(s)=" + f"{nb:.6f}"))
        display(Math(r"G_{\mathrm{Pois}}(s)=" + f"{pois:.6f}"))


pgf_s.observe(update_pgf, names="value")
display(widgets.VBox([pgf_s, pgf_output]))
update_pgf()


## 15. Choosing a count distribution

A named law is not justified merely because its parameters can be fitted.

Model choice should consider:

- support;
- experiment or sampling mechanism;
- independence or dependence;
- homogeneity of success probabilities or event rates;
- observation scale;
- mean--variance behavior;
- zero frequency and tails.

Dispersion is a useful first diagnostic, not a proof of model correctness.


### Dispersion signatures

On positive-mean non-degenerate parameter ranges:

$$
\operatorname{Bin}(n,p):
\qquad
D=1-p<1,
$$

$$
\operatorname{Hyp}(N,K,n):
\qquad
D=
\left(1-\frac KN\right)
\frac{N-n}{N-1}<1,
$$

$$
\operatorname{Poisson}(\lambda):
\qquad
D=1,
$$

$$
\operatorname{Geom}(p):
\qquad
D=\frac1p>1,
$$

$$
\operatorname{NB}(r,p):
\qquad
D=\frac1p>1.
$$


In [ ]:
model_mean = widgets.FloatSlider(value=8, min=0.5, max=20, step=0.5, description="mean")
model_var = widgets.FloatSlider(value=16, min=0.1, max=40, step=0.5, description="variance")
model_output = widgets.Output()


def update_model_diagnostic(*_):
    with model_output:
        clear_output(wait=True)

        m = model_mean.value
        v = model_var.value
        D = v/m

        display(Math(r"D=" + f"{D:.4f}"))

        if D < 1 - 1e-8:
            display(Markdown(
                "**Moment signature:** underdispersed. "
                "Binomial or hypergeometric families may be candidates, "
                "depending on the mechanism."
            ))
        elif D > 1 + 1e-8:
            display(Markdown(
                "**Moment signature:** overdispersed. "
                "A negative binomial family may be a candidate; "
                "a waiting-time interpretation is not automatic."
            ))
        else:
            display(Markdown(
                "**Moment signature:** equidispersed. "
                "Poisson is compatible with this moment relation, "
                "but empirical equidispersion does not prove a Poisson mechanism."
            ))


for control in (model_mean, model_var):
    control.observe(update_model_diagnostic, names="value")

display(widgets.VBox([
    widgets.HBox([model_mean, model_var]),
    model_output,
]))
update_model_diagnostic()


### Structural comparison

| Model | Natural mechanism | Main warning |
|---|---|---|
| Bernoulli | one yes/no event | cannot represent counts larger than one |
| Binomial | fixed number of homogeneous independent trials | heterogeneity or dependence breaks the construction |
| Hypergeometric | uniform sampling without replacement | not a repeated-independent-trial model |
| Poisson | homogeneous rare-event count over a fixed observation scale | mean--variance equality may be too restrictive |
| Geometric | failures before a first success | waiting-time interpretation must be appropriate |
| Negative binomial | failures before the $r$-th success or flexible overdispersed count | parameterization and source of overdispersion must be stated |

> Matching mean and variance is a diagnostic starting point, not a model validation.


## 16. Stable Poisson computation

Direct factorial formulas can overflow, while $e^{-\lambda}$ can underflow when $\lambda$ is large.

A more stable strategy is:

1. compute a mass near the mode using logarithms;
2. recurse upward using

$$
p(k+1)=\frac{\lambda}{k+1}p(k);
$$

3. recurse downward using

$$
p(k-1)=\frac{k}{\lambda}p(k).
$$


In [ ]:
def poisson_table_stable(lam, width=8):
    if isinstance(lam, bool) or lam <= 0:
        raise ValueError("lam must be positive.")
    mode = int(floor(lam))
    low = max(0, mode-width)
    high = mode+width

    log_p_mode = -lam + mode*math.log(lam) - lgamma(mode+1)
    probs = {mode: math.exp(log_p_mode)}

    for k in range(mode, high):
        probs[k+1] = probs[k] * lam/(k+1)

    for k in range(mode, low, -1):
        probs[k-1] = probs[k] * k/lam

    ks = np.arange(low, high+1)
    ps = np.array([probs[int(k)] for k in ks])
    return ks, ps


stable_lam = widgets.FloatSlider(value=100, min=5, max=500, step=5, description="lambda")
stable_output = widgets.Output()


def update_stable_poisson(*_):
    with stable_output:
        clear_output(wait=True)

        lam = stable_lam.value
        ks, ps = poisson_table_stable(lam, width=10)

        fig, ax = plt.subplots(figsize=(8, 3.4))
        ax.stem(ks, ps)
        ax.set_xlabel("k near mode")
        ax.set_ylabel("mass")
        ax.set_title("Stable local Poisson table around the mode")
        plt.show()

        display(Markdown(
            f"Mass near mode $k={int(floor(lam))}$: **{ps[10]:.6g}**"
        ))


stable_lam.observe(update_stable_poisson, names="value")
display(widgets.VBox([stable_lam, stable_output]))
update_stable_poisson()


## 17. Solved-style computational examples


### Binomial trials and Poisson approximation

For

$$
N\sim\operatorname{Bin}(100,0.02),
$$

$$
\mathbb E[N]=2,
\qquad
\operatorname{Var}(N)=1.96.
$$

The exact probabilities are

$$
P(N=0)=0.98^{100},
$$

$$
P(N\ge2)
=
1-0.98^{100}
-
100(0.02)(0.98)^{99}.
$$

The Poisson approximation uses $Z\sim\operatorname{Poisson}(2)$.


In [ ]:
p0_bin = binomial_pmf(0, 100, 0.02)
p_ge2_bin = 1 - binomial_pmf(0,100,0.02) - binomial_pmf(1,100,0.02)

p0_pois = poisson_pmf(0, 2)
p_ge2_pois = 1 - poisson_pmf(0,2) - poisson_pmf(1,2)

display(Math(r"P_{\mathrm{Bin}}(0)=" + f"{p0_bin:.6f}"))
display(Math(r"P_{\mathrm{Bin}}(N\ge2)=" + f"{p_ge2_bin:.6f}"))
display(Math(r"P_{\mathrm{Pois}}(0)=" + f"{p0_pois:.6f}"))
display(Math(r"P_{\mathrm{Pois}}(Z\ge2)=" + f"{p_ge2_pois:.6f}"))


### Geometric memorylessness used correctly

If

$$
X\sim\operatorname{Geom}(0.25),
$$

then, given at least five failures,

$$
P(X\ge8\mid X\ge5)
=
P(X\ge3)
=
0.75^3
=
0.421875.
$$

The expected total number of trials is

$$
\mathbb E[T]=\frac1{0.25}=4.
$$


In [ ]:
display(Math(
    r"P(X\ge8\mid X\ge5)=" + f"{0.75**3:.6f}"
))
display(Math(r"\mathbb E[T]=4"))


### Negative binomial calibration

Target mean $6$ and variance $12$ give

$$
p=\frac6{12}=\frac12,
$$

$$
r=\frac{6^2}{12-6}=6.
$$

Thus

$$
N\sim\operatorname{NB}(6,1/2),
$$

and

$$
P(N=0)=\left(\frac12\right)^6=\frac1{64}.
$$


In [ ]:
display(Math(
    r"P(N=0)=" + fmt_fraction(Fraction(1,64))
))


### Hypergeometric audit

From $100$ objects, $20$ are marked and $10$ are sampled without replacement.

Then

$$
X\sim\operatorname{Hyp}(100,20,10).
$$

The mean is

$$
\mathbb E[X]=2,
$$

and

$$
\operatorname{Var}(X)
=
10(0.2)(0.8)\frac{90}{99}
=
\frac{16}{11}.
$$


In [ ]:
p0 = hypergeometric_pmf(0, 100, 20, 10)
p_ge1 = 1-p0
var = 10*0.2*0.8*90/99

display(Math(r"P(X=0)=" + f"{p0:.8f}"))
display(Math(r"P(X\ge1)=" + f"{p_ge1:.8f}"))
display(Math(r"\operatorname{Var}(X)=" + f"{var:.8f}"))


### Poisson tail from recursion

For

$$
N\sim\operatorname{Poisson}(3),
$$

start with

$$
p_0=e^{-3}.
$$

Then

$$
p_1=3e^{-3},
$$

$$
p_2=\frac92e^{-3}.
$$

Therefore

$$
P(N\le2)
=
\frac{17}{2}e^{-3},
$$

and

$$
P(N\ge3)
=
1-\frac{17}{2}e^{-3}.
$$


In [ ]:
p_le2 = sum(poisson_pmf(k,3) for k in range(3))
display(Math(r"P(N\le2)=" + f"{p_le2:.8f}"))
display(Math(r"P(N\ge3)=" + f"{1-p_le2:.8f}"))


## 18. Guided exercise generator


In [ ]:
exercise_rng = random.Random(20260815)

exercise_kind = widgets.Dropdown(
    options=[
        ("Random", "random"),
        ("Pmf", "pmf"),
        ("Variance", "variance"),
        ("Binomial", "binomial"),
        ("Geometric", "geometric"),
        ("Negative binomial", "nb"),
        ("Hypergeometric", "hyper"),
        ("Poisson", "poisson"),
        ("Pgf", "pgf"),
        ("Model choice", "model"),
    ],
    value="random",
    description="Type",
)

new_button = widgets.Button(description="New exercise")
hint_button = widgets.Button(description="Hint")
reveal_button = widgets.Button(description="Reveal")
check_button = widgets.Button(description="Check")
answer_box = widgets.Text(description="Answer")
prompt_output = widgets.Output()
feedback_output = widgets.Output()
state = {}


def make_exercise(_=None):
    kind = exercise_kind.value

    if kind == "random":
        kind = exercise_rng.choice([
            "pmf","variance","binomial","geometric","nb",
            "hyper","poisson","pgf","model"
        ])

    if kind == "pmf":
        target = "2/3"
        prompt = "If p(k)=2^{-(k+1)} for k>=0, what is P(X is even)? Enter 2/3."
        hint = "Sum a geometric series over k=0,2,4,..."
        solution = r"P(X\text{ even})=\frac23."

    elif kind == "variance":
        target = "0.61"
        prompt = "P(N=0)=0.5, P(N=1)=0.3, P(N=2)=0.2. Find Var(N)."
        hint = "Use E[(N)_2]+E[N]-(E[N])^2."
        solution = r"\operatorname{Var}(N)=0.61."

    elif kind == "binomial":
        target = "10"
        prompt = "For Bin(500,0.02), find the mean."
        hint = "Use np."
        solution = r"\mathbb E[X]=500(0.02)=10."

    elif kind == "geometric":
        target = "3"
        prompt = "For Geom(0.25), counting failures before first success, find E[X]."
        hint = "Use (1-p)/p."
        solution = r"\mathbb E[X]=3."

    elif kind == "nb":
        target = "2"
        prompt = "For NB(4,1/2), what is the dispersion index?"
        hint = "For this family D=1/p."
        solution = r"D=2."

    elif kind == "hyper":
        target = "2"
        prompt = "For Hyp(100,20,10), find E[X]."
        hint = "Use nK/N."
        solution = r"\mathbb E[X]=2."

    elif kind == "poisson":
        target = "1"
        prompt = "What is the dispersion index of a positive-mean Poisson law?"
        hint = "Mean equals variance."
        solution = r"D=1."

    elif kind == "pgf":
        target = "yes"
        prompt = "Does the pgf uniquely determine a non-negative integer-valued law? yes/no"
        hint = "Its derivatives at zero recover every pmf coefficient."
        solution = r"\text{Yes.}"

    else:
        target = "no"
        prompt = "If empirical mean equals empirical variance, does this prove a Poisson model? yes/no"
        hint = "Moment compatibility does not verify the full mechanism or tail fit."
        solution = r"\text{No.}"

    state.clear()
    state.update(
        target=target,
        hint=hint,
        solution=solution,
    )
    answer_box.value = ""

    with prompt_output:
        clear_output(wait=True)
        display(Markdown("### Exercise\n" + prompt))

    with feedback_output:
        clear_output(wait=True)


def show_hint(_):
    with feedback_output:
        clear_output(wait=True)
        display(Markdown("**Hint:** " + state["hint"]))


def reveal(_):
    with feedback_output:
        clear_output(wait=True)
        display(Math(state["solution"]))


def check(_):
    with feedback_output:
        clear_output(wait=True)

        guess = answer_box.value.strip().lower().replace(" ", "")
        target = state["target"].replace(" ", "")

        numeric_ok = False
        try:
            if "/" in target:
                a, b = target.split("/")
                numeric_ok = abs(float(guess) - float(a)/float(b)) < 1e-8
            elif target not in {"yes","no"}:
                numeric_ok = abs(float(guess) - float(target)) < 1e-8
        except Exception:
            pass

        if guess == target or numeric_ok:
            display(Markdown("**Correct.**"))
        else:
            display(Markdown(
                "**Not yet.** Check the support convention and the structural formula first."
            ))


new_button.on_click(make_exercise)
hint_button.on_click(show_hint)
reveal_button.on_click(reveal)
check_button.on_click(check)

display(widgets.VBox([
    widgets.HBox([exercise_kind, new_button]),
    prompt_output,
    widgets.HBox([answer_box, check_button]),
    widgets.HBox([hint_button, reveal_button]),
    feedback_output,
]))

make_exercise()


## 19. AI Audit: discrete laws and count models

Use this checklist on any AI-generated solution.

1. Is the distinction between **countably valued** and **discrete law** respected?
2. Do proposed pmf values sum to one?
3. Is the pmf support being confused with topological support?
4. Is a finite expectation being assumed merely because the law is discrete?
5. Are variance and standard deviation being confused?
6. Is a count model selected from mean and variance alone?
7. Does a binomial argument genuinely use a fixed number of homogeneous mutually independent trials?
8. Is pairwise independence incorrectly treated as sufficient for arbitrary success/failure pattern probabilities?
9. Is the geometric support convention stated?
10. Is memorylessness being applied to the correct waiting-time variable?
11. Is the negative binomial parameterization stated explicitly?
12. Is $r$ being treated as an integer under this chapter's waiting-time convention?
13. Is hypergeometric sampling correctly identified as **without replacement**?
14. Is the finite population correction present in the hypergeometric variance?
15. Is Poisson equidispersion being treated as a model implication rather than a universal law for counts?
16. Is a Poisson approximation being confused with exact equality?
17. Is the Poisson limit theorem being used outside its rare-event scaling?
18. Does the pgf use the correct convention at $s=0$?
19. Is $G_X^{(r)}$ correctly interpreted as the $r$-th derivative?
20. Are factorial moments recovered from the left limit at $1$ when needed?
21. Is a numerically unstable factorial computation being used for large parameters when a recursion or log-mass method is preferable?

### Claims to audit

- “Pairwise independence is always sufficient for the binomial derivation.”
- “Hypergeometric sampling is just binomial sampling with a different formula.”
- “If a dataset has mean equal to variance, the data must be Poisson.”
- “If $np=\lambda$, then $\operatorname{Bin}(n,p)$ and $\operatorname{Poisson}(\lambda)$ are identical.”

All four claims are false.


### Suggested AI-guided activities

- “Give me a finite pmf and ask me to verify normalization, construct the cdf and compute the first two factorial moments before deriving the variance.”
- “Create underdispersed, equidispersed and overdispersed count datasets with the same mean, then challenge every proposed model by asking about mechanism and tails.”
- “Guide me through the Poisson limit proof one factor at a time and reject any unjustified interchange of an infinite sum and a limit.”
- “Give me a pgf, make me recover the pmf from derivatives at zero, then compute factorial moments from derivatives near one.”
- “Give me geometric and negative-binomial word problems with ambiguous conventions and require me to state the support before solving.”


## 20. Self-check quiz


In [ ]:
quiz_data = [
    (
        "1. A discrete law is:",
        [
            "Choose...",
            "a pointwise property of X(Omega)",
            "a property that the law is concentrated on a countable set",
        ],
        "a property that the law is concentrated on a countable set",
        r"\text{Discrete-law status is a property of the probability law.}",
    ),
    (
        "2. A pmf must:",
        [
            "Choose...",
            "sum to one",
            "be strictly positive everywhere",
            "have finite support",
        ],
        "sum to one",
        r"\sum_x p_X(x)=1.",
    ),
    (
        "3. Var(aX+b) equals:",
        [
            "Choose...",
            "a Var(X)+b",
            "a^2 Var(X)",
            "Var(X)+b^2",
        ],
        "a^2 Var(X)",
        r"\operatorname{Var}(aX+b)=a^2\operatorname{Var}(X).",
    ),
    (
        "4. The binomial model requires:",
        [
            "Choose...",
            "fixed homogeneous mutually independent trials",
            "sampling without replacement",
            "only equal means",
        ],
        "fixed homogeneous mutually independent trials",
        r"\text{Those are the structural assumptions behind the binomial derivation.}",
    ),
    (
        "5. In this chapter Geom(p) counts:",
        [
            "Choose...",
            "failures before first success",
            "total trials including first success",
        ],
        "failures before first success",
        r"\text{The support begins at zero.}",
    ),
    (
        "6. Under this convention NB(1,p) equals:",
        ["Choose...", "Geom(p)", "Poisson(p)", "Bernoulli(p)"],
        "Geom(p)",
        r"\operatorname{NB}(1,p)=\operatorname{Geom}(p).",
    ),
    (
        "7. Hypergeometric sampling is:",
        [
            "Choose...",
            "with replacement",
            "without replacement",
        ],
        "without replacement",
        r"\text{Dependence produces the finite population correction.}",
    ),
    (
        "8. A positive-mean Poisson law is:",
        [
            "Choose...",
            "underdispersed",
            "equidispersed",
            "overdispersed",
        ],
        "equidispersed",
        r"\operatorname{Var}(X)=\mathbb E[X]=\lambda.",
    ),
    (
        "9. The Poisson approximation to a finite binomial law is:",
        [
            "Choose...",
            "always exact",
            "an approximation under rare-event conditions",
        ],
        "an approximation under rare-event conditions",
        r"\text{Matching }np=\lambda\text{ does not imply equality of laws.}",
    ),
    (
        "10. A pgf uniquely determines a non-negative integer-valued law:",
        ["Choose...", "true", "false"],
        "true",
        r"p_X(k)=G_X^{(k)}(0)/k!.",
    ),
    (
        "11. G_X^{(r)} denotes:",
        [
            "Choose...",
            "the r-th derivative",
            "the r-th power",
        ],
        "the r-th derivative",
        r"G_X^{(r)}\text{ denotes the }r\text{-th derivative.}",
    ),
    (
        "12. Mean/variance matching alone validates a count model:",
        ["Choose...", "true", "false"],
        "false",
        r"\text{Mechanism, support and tail fit must also be examined.}",
    ),
]

quiz_widgets = []
quiz_rows = []

for prompt, options, _, _ in quiz_data:
    d = widgets.Dropdown(
        options=options,
        value="Choose...",
        layout=widgets.Layout(width="470px"),
    )
    quiz_widgets.append(d)
    quiz_rows.append(widgets.HBox([
        widgets.HTML(f"<div style='width:650px'>{prompt}</div>"),
        d,
    ]))

grade_button = widgets.Button(description="Grade quiz")
quiz_output = widgets.Output()


def grade_quiz(_):
    with quiz_output:
        clear_output(wait=True)

        score = sum(
            widget.value == correct
            for widget, (_, _, correct, _) in zip(
                quiz_widgets,
                quiz_data,
            )
        )

        display(Markdown(f"### Score: {score}/{len(quiz_data)}"))

        for i, (
            widget,
            (_, _, correct, explanation),
        ) in enumerate(zip(quiz_widgets, quiz_data), 1):
            mark = "✓" if widget.value == correct else "✗"
            display(Markdown(
                f"**{mark} Question {i}:** correct answer = `{correct}`"
            ))
            display(Math(explanation))


grade_button.on_click(grade_quiz)
display(widgets.VBox(
    quiz_rows + [grade_button, quiz_output]
))


## 21. Automatic mathematical verification

This final cell checks normalization, theoretical moments, memorylessness, Newton--Pepys ordering, pgfs and one Poisson-approximation convergence diagnostic.


In [ ]:
# Discrete uniform.
m = 7
ks = np.arange(1, m+1)
probs = np.array([discrete_uniform_pmf(int(k), m) for k in ks])
total, mean, variance = moments(ks, probs)
assert np.isclose(total, 1)
assert np.isclose(mean, (m+1)/2)
assert np.isclose(variance, (m*m-1)/12)

# Bernoulli.
p = 0.3
ks = np.array([0,1])
probs = np.array([bernoulli_pmf(int(k), p) for k in ks])
_, mean, variance = moments(ks, probs)
assert np.isclose(mean, p)
assert np.isclose(variance, p*(1-p))

# Binomial.
n, p = 20, 0.10
ks = np.arange(n+1)
probs = np.array([binomial_pmf(int(k), n, p) for k in ks])
total, mean, variance = moments(ks, probs)
assert np.isclose(total, 1)
assert np.isclose(mean, n*p)
assert np.isclose(variance, n*p*(1-p))

# Geometric truncated normalization and theoretical moments.
p = 0.25
q = 1-p
assert np.isclose(q**8/q**5, q**3)
assert np.isclose(q/p, 3)
assert np.isclose(q/(p*p), 12)

# Negative binomial special case.
for k in range(20):
    assert np.isclose(
        negative_binomial_pmf(k, 1, 0.2),
        geometric_pmf(k, 0.2),
    )

# Negative binomial moments.
r, p = 4, 0.5
assert np.isclose(r*(1-p)/p, 4)
assert np.isclose(r*(1-p)/(p*p), 8)

# Hypergeometric.
N, K, n = 100, 20, 10
lower = max(0, n-(N-K))
upper = min(n, K)
ks = np.arange(lower, upper+1)
probs = np.array([
    hypergeometric_pmf(int(k), N, K, n)
    for k in ks
])
total, mean, variance = moments(ks, probs)
theory_var = n*(K/N)*(1-K/N)*(N-n)/(N-1)

assert np.isclose(total, 1)
assert np.isclose(mean, n*K/N)
assert np.isclose(variance, theory_var)

# Poisson truncated check at lambda=4.
lam = 4
ks = np.arange(0, 40)
probs = np.array([poisson_pmf(int(k), lam) for k in ks])
total, mean, variance = moments(ks, probs)

assert np.isclose(total, 1, atol=1e-12)
assert np.isclose(mean, lam, atol=1e-10)
assert np.isclose(variance, lam, atol=1e-9)

# Poisson recursion.
for k in range(10):
    lhs = poisson_pmf(k+1, lam)
    rhs = lam/(k+1) * poisson_pmf(k, lam)
    assert np.isclose(lhs, rhs)

# Newton-Pepys.
pepys = []
for r in (1,2,3):
    prob = sum(
        binomial_pmf(k, 6*r, 1/6)
        for k in range(r, 6*r+1)
    )
    pepys.append(prob)

assert pepys[0] > pepys[1] > pepys[2]

# PGF checks at s=0.7.
s = 0.7

# Binomial
pgf_numeric = np.sum(
    np.array([binomial_pmf(k, 5, 0.2) for k in range(6)])
    * s**np.arange(6)
)
assert np.isclose(pgf_numeric, (0.8+0.2*s)**5)

# Poisson approximation diagnostic.
lam = 2
sample_sizes = np.array([20,100,500])
zero_probs = np.array([
    (1-lam/n)**n
    for n in sample_sizes
])
errors = np.abs(zero_probs - exp(-lam))
assert np.all(np.diff(errors) < 0)

# Stable Poisson table has positive values.
stable_ks, stable_ps = poisson_table_stable(100, width=8)
assert np.all(stable_ps > 0)

show_result(
    "All Chapter 8 automatic checks passed",
    r"\sum_k p_X(k)=1",
    r"\operatorname{Var}(X)=\mathbb E[X^2]-\mathbb E[X]^2",
    r"\operatorname{NB}(1,p)=\operatorname{Geom}(p)",
    r"\operatorname{Var}(\operatorname{Poisson}(\lambda))=\lambda",
    r"p_X(k+1)=\frac{\lambda}{k+1}p_X(k)",
    r"p_X(k)=\frac{G_X^{(k)}(0)}{k!}",
    note=(
        "Normalization, moments, memorylessness, Newton--Pepys ordering, "
        "PGFs, hypergeometric correction and Poisson approximation checks all passed."
    ),
)


## 22. Chapter map

| Chapter concept | Computational representation |
|---|---|
| countably valued vs discrete law | same map under different probability measures |
| pmf | normalization and event sums |
| cdf from pmf | cumulative probability table |
| expectation from pmf | weighted count sum |
| raw/central/factorial moments | direct finite computations |
| variance | computational formula and affine transformation |
| dispersion index | interactive mean--variance diagnostic |
| discrete uniform | pmf and moment verification |
| Bernoulli | indicator law |
| binomial | homogeneous independent trial count |
| Newton--Pepys | exact binomial tails |
| geometric | waiting time and memorylessness |
| negative binomial | failures before $r$-th success and overdispersion |
| hypergeometric | sampling without replacement and finite population correction |
| Poisson | recursion, mode and equidispersion |
| Poisson limit | rare-event binomial convergence |
| pgf | law encoding and coefficient recovery |
| pgf derivatives | factorial moments |
| model selection | mechanism plus dispersion and tail diagnostics |
| numerical stability | log-mass plus recursion near Poisson mode |
| AI Audit | structural and parameterization checks |

The central modelling lesson is:

> A familiar pmf formula is not a substitute for a probability model. Support, mechanism, dependence assumptions, parameterization and tail behavior must all fit the experiment.
